# Element of a Prompt

**Instruction:** A specific task or instruction you want the model to perform.

**Context:** External Information or additional context that can steer the model to a better response.

**Input data:** The input or question that we are interested in finding a response for

## Trying It With Gemini

These examples use [Google Gemini's free tier](https://ai.google.dev/gemini-api/docs/rate-limits) — genuinely free, no card required. `gemini-flash-lite-latest` is Google's lightest, fastest model, a good fit for free-tier rate limits.

It reads `GEMINI_API_KEY` from `.env` via the `google-genai` SDK (`pip install google-genai`). One client, one `generate_content` call — settings like `temperature`, `top_p`, `max_output_tokens`, and `stop_sequences` all live inside a `GenerateContentConfig`.

### Temperature & Top_p

Both settings act on the probability distribution over the *next token*, before one is sampled:
- **Temperature** sharpens (low) or flattens (high) that distribution. Low temperature means the model almost always picks its single most-likely next token.
- **Top_p** shrinks (low) or widens (high) the *set* of tokens eligible to be sampled at all.

"Low temperature = deterministic" doesn't mean the answer is short — it means the same prompt produces the same output every time you run it. Response length is a separate thing, controlled by `max_output_tokens` (below) and by however long the model's own most-likely continuation happens to be.

To actually see the effect, the cell below sends the *same* prompt three times at low temperature/top_p, then three times at high temperature/top_p.

In [9]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
prompt = "Describe the sky in one sentence."

print("--- Low temperature (0.1) + low top_p (0.1), same prompt run 3x ---")
for i in range(3):
    response = gemini_client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1, top_p=0.1, max_output_tokens=1024),
    )
    print(f"{i + 1}:", response.text)

print("\n--- High temperature (1.0) + high top_p (0.98), same prompt run 3x ---")
for i in range(3):
    response = gemini_client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=1.0, top_p=0.98, max_output_tokens=1024),
    )
    print(f"{i + 1}:", response.text)

--- Low temperature (0.1) + low top_p (0.1), same prompt run 3x ---
1: The sky is an infinite, shifting canvas that transitions from the brilliant, sun-drenched azure of day to the velvet, star-dusted expanse of night.
2: The sky is an infinite, shifting canvas that transitions from the brilliant, sun-drenched azure of day to the velvet, star-dusted expanse of night.
3: The sky is an infinite, shifting canvas that transitions from the brilliant, sun-drenched azure of day to the velvet, star-dusted expanse of night.

--- High temperature (1.0) + high top_p (0.98), same prompt run 3x ---
1: The sky is a vast, shifting canvas that mirrors the moods of the earth, transitioning from the piercing brilliance of a sun-drenched day to the velvet, star-dusted abyss of the night.
2: The sky is an infinite canvas of shifting light, stretching from the tranquil gradients of dawn to the vast, star-strewn velvet of the night.
3: The sky is an infinite canvas of shifting light, stretching from the dee

## Stop Sequence

A **stop sequence** tells the model exactly where to stop generating text — generation halts the instant the model produces the given sequence, and that sequence itself is excluded from the returned text.

This is useful for keeping output bounded — for example, generating a fixed-size list without trailing commentary.

In [15]:
gemini_prompt = "List exactly three fruits separated by commas, then write STOP."

# Without a stop sequence: the model's own "STOP" text comes back in the output
response = gemini_client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=gemini_prompt,
    config=types.GenerateContentConfig(temperature=0.1, top_p=0.1, max_output_tokens=256),
)
print("Without stop sequence:")
print(response.text)

# With stop_sequences: generation halts the moment "STOP" would appear, and STOP itself is dropped from the output
response = gemini_client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=gemini_prompt,
    config=types.GenerateContentConfig(
        temperature=0.1, top_p=0.1, max_output_tokens=256, stop_sequences=["STOP"]
    ),
)
print("\nWith stop sequence:")
print(response.text)

Without stop sequence:
Apple, banana, orange STOP

With stop sequence:
Apple, banana, orange


## Zero-Shot Prompting

Just ask — no examples needed.

- **Problem it fixes:** it's the fastest way to ask for something.
- **When it works:** the task is common and easy — sentiment, translation, simple questions.
- **When it fails:** the task is unusual, or you need a specific format. The model just guesses, and its guesses won't always match.

See [2_prompt_engineering.md](2_prompt_engineering.md) for the rest of the techniques (few-shot, chain-of-thought, ReAct, and more) — this cell just proves the first one actually works.

In [14]:
# zero_shot_prompt = 'Classify this review as positive or negative: "The battery life is terrible."'

zero_shot_prompt = 'Classify NEG or POS: "Broke after one day."'

response = gemini_client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=zero_shot_prompt,
    config=types.GenerateContentConfig(temperature=0.1, top_p=0.1, max_output_tokens=256),
)

print(response.text)

NEG


## Chain-of-Thought (CoT)

Ask the model to think step by step, instead of jumping straight to an answer.

- **Problem it fixes:** the model writes one word at a time, with no scratch paper. On a hard problem, jumping straight to the answer skips the real thinking. Writing it out step by step gives it a place to work things out first.
- **When to use it:** math, logic, anything with more than one step.
- **Cost:** more words means more time and money. Newer "thinking" models already do a version of this on their own.

The cell below asks the same word problem two ways — once demanding just a number, once asking it to think step by step — so you can see the difference in the raw output.

In [16]:
no_cot_prompt = "A store had 23 apples, sold 15, then got 8 more. How many now? Answer with just the number."
cot_prompt = "A store had 23 apples, sold 15, then got 8 more. Think step by step, then give the final answer."

response = gemini_client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=no_cot_prompt,
    config=types.GenerateContentConfig(temperature=0.1, top_p=0.1, max_output_tokens=256),
)
print("Without CoT:")
print(response.text)

response = gemini_client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=cot_prompt,
    config=types.GenerateContentConfig(temperature=0.1, top_p=0.1, max_output_tokens=512),
)
print("\nWith CoT:")
print(response.text)

Without CoT:
16

With CoT:
Here is the step-by-step breakdown:

1.  **Starting amount:** The store had 23 apples.
2.  **After selling:** 23 - 15 = 8 apples remaining.
3.  **After getting more:** 8 + 8 = 16 apples.

**Final Answer:** The store has 16 apples.


## ReAct (Reason + Act)

Think out loud, take an action, look at what happened, then think again — on repeat until you're done.

- **Problem it fixes:** plain step-by-step thinking can't check real facts, so it can guess wrong and still sound confident. Plain tool-calling can check facts, but it doesn't explain *why* it's calling a tool, so it can pick the wrong one or misread the result. ReAct does both at once: think, act, check, think again.
- **When to use it:** this is the pattern behind almost every AI "agent" that uses tools today — including the assistant helping you right now.
- **Where the full version lives:** here, you're learning it as a way of prompting. The real version — actual tools, memory across steps, and what to do when a tool fails — is covered in [4_applied_ai](../4_applied_ai/README.md). Come back and re-read this section once you've built one; it'll click differently.

The cell below wires up a tiny real `search` tool (just a hardcoded lookup, standing in for a real web search) and drives the Thought → Action → Observation loop by hand: the same `stop_sequences` trick from earlier stops the model right before it would fabricate its own Observation, then the real result gets spliced in and fed back for the next step.

In [ ]:
import re


def search(query: str) -> str:
    """Toy search tool — a tiny hardcoded knowledge base standing in for a real web search."""
    q = query.lower()
    if "eiffel" in q:
        return "France"
    if "capital" in q and "france" in q:
        return "Paris"
    if "population" in q and "paris" in q:
        return "about 2.1 million"
    return "No information found."


react_priming = """You have no built-in knowledge. You must call search("<query>") to look up every single fact, one fact per Action, before using it in your reasoning. Never state a fact you have not just looked up. Follow this exact format:

Question: <question>
Thought: <your reasoning>
Action: search("<query>")
Observation: <result will be filled in>
... (repeat Thought/Action/Observation as needed)
Thought: I have everything I need.
Final Answer: <answer>

Question: What's the population of the capital of the country the Eiffel Tower is in?
"""

transcript = react_priming

for step in range(5):
    response = gemini_client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=transcript,
        config=types.GenerateContentConfig(
            temperature=0.1, top_p=0.1, max_output_tokens=256, stop_sequences=["Observation:"]
        ),
    )
    transcript += response.text
    if "Final Answer:" in response.text:
        break
    match = re.search(r'Action:\s*search\(["\']([^"\']+)["\']\)', response.text)
    if not match:
        break
    observation = search(match.group(1))
    transcript += f"Observation: {observation}\n"

print(transcript)